# 02 — Isolation Forest
Train Isolation Forest on processed train set, score test set, write `submission_isolation_forest.csv`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yaml
from src.features import feature_names
from src.models import score_isolation_forest

cfg = yaml.safe_load((ROOT / 'configs' / 'config.yaml').read_text())
cols = feature_names(cfg['features']['raw'])

In [ ]:
proc = ROOT / 'data' / 'processed'
train = pd.read_parquet(proc / 'train.parquet')
test = pd.read_parquet(proc / 'test.parquet')

def mat(df):
    X = df[cols].to_numpy(dtype=np.float32, na_value=0.0)
    np.nan_to_num(X, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return X

X_tr, X_te = mat(train), mat(test)

In [ ]:
scores = score_isolation_forest(X_tr, X_te, cfg['models']['isolation_forest'], cfg['seed'])
scores.min(), scores.max(), scores.mean()

In [ ]:
out = ROOT / 'outputs' / 'submissions'
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame({'ID': test['index'], 'TARGET': scores}).to_csv(out / 'submission_isolation_forest.csv', index=False)